# 02: Spiking Dynamics & Multi-Timescale Memory
Analysis of membrane voltage dynamics, surrogate gradient backpropagation, and multi-timescale integration.


In [ ]:
import torch
import matplotlib.pyplot as plt
from spwm.models.memory import MultiTimescaleMemory
from spwm.models.surrogate import get_surrogate

# Check surrogate gradients
x = torch.linspace(-2.0, 2.0, 200, requires_grad=True)
atan_surr = get_surrogate("atan", alpha=2.0)
s = atan_surr(x)
loss = s.sum()
loss.backward()

plt.figure(figsize=(6, 3))
plt.plot(x.detach().numpy(), x.grad.numpy(), label="Atan Surrogate Derivative dS/dx")
plt.title("Surrogate Gradient Profile")
plt.xlabel("V - V_th"); plt.legend(); plt.show()


In [ ]:
# Multi-timescale decay comparison
mem = MultiTimescaleMemory(timescale_dims=(1, 1), betas=(0.7, 0.98))
state = mem.init_state(1)
_, state = mem(torch.tensor([[5.0, 5.0]]), state)

fast_trace, slow_trace = [state.v_mems[0].item()], [state.v_mems[1].item()]
for _ in range(25):
    _, state = mem(torch.zeros(1, 2), state)
    fast_trace.append(state.v_mems[0].item())
    slow_trace.append(state.v_mems[1].item())

plt.figure(figsize=(8, 4))
plt.plot(fast_trace, label="Fast Memory (β=0.70)")
plt.plot(slow_trace, label="Slow Memory (β=0.98)")
plt.title("Decay Dynamics: Fast vs Slow Spiking Populations")
plt.xlabel("Timesteps"); plt.ylabel("Membrane Potential"); plt.legend(); plt.show()
